In [ ]:
%load_ext autoreload
%autoreload 2

%gui qt

In [ ]:
import os
import sys
import importlib
import cv2
gui_source_dir = os.path.expanduser("~/Documents/talmolab/repos/lucid_lite/gui_source")
josh_source_dir = os.path.expanduser("~/Documents/talmolab/repos/lucid_lite")

# add folders to Python's system path so it can find local imports
if gui_source_dir not in sys.path:
    sys.path.insert(0, gui_source_dir)
    sys.path.insert(0, josh_source_dir)

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import numpy as np
import pandas as pd
import sleap_io as sio

from PySide6.QtGui import QImage


from lucid_lite.gui_source import main
from lucid_lite.gui_source import colors

from josh_source import epipolar
from josh_source import geometry
from josh_source import tracker

In [ ]:
app, window = main.main(
    [
        "main.py",                                                          
        "/Users/joshuapark/Documents/talmolab/lucid_folders/10072022145420_small"
    ]
)
# define globals
session = window.session

SKEL = session.skeleton
TRACK_COLOR_MAP = {track: colors.get_track_color(n) for n, track in enumerate(session.tracks)}
CAMERAS = {camera.name: camera for camera in session.cameras}

NODE_NAMES = [
    "Nose",
    "Ear_R",
    "Ear_L",
    "TTI",
    "TailTip",
    "Head",
    "Trunk",
    "Tail_0",
    "Tail_1",
    "Tail_2",
    "Shoulder_left",
    "Shoulder_right",
    "Haunch_left",
    "Haunch_right",
    "Neck"
]

def frame_df(frame_idx = None):
    if frame_idx is None:
        fg = session.frame_group(window._current_frame)
    else:
        fg = session.frame_group(frame_idx)
    rows = []
    for cam, insts in fg.instances.items():
        for inst in insts:
            rows.append({
                'cam': cam,
                'points': inst.points,
                'track_idx': inst.track_idx,
                "track": (session.tracks[inst.track_idx]
                        if inst.track_idx is not None else None),
                "identity_id": session.get_identity_id_for_track(
                    window._current_frame, cam, inst.track_idx),
                "type": inst.type,
                "score": inst.score,
                "n_visible": sum(p is not None for p in inst.points),
            })
    return pd.DataFrame(rows)


In [ ]:
'''
tracker flaws:

1. everything depends on the first and second 'pivot' frames, which
    are determined by the number of instances in the view, nothing
    about the quality of the views
2. if there is a singleton group from the first two frames, then
    they remain as a singleton group
3. order-dependent recruitment: the ordering of the cameras that are
    seen after the first 2 views also matters in terms of adding to the group
        eg. cam3 then cam4 is seen, cam4's addition to the same group
        can be influenced by cam3's additions
    - subpoint: NO ALGORITHM TO DETERMINE THE ORDER OF CAMERAS
4. why do hungarian on negative of score matrix???
5. reproj error: not using the distored points

'''

In [ ]:
importlib.reload(tracker)


curr_frame = window._current_frame
fg = session.frame_group(curr_frame)
cam_instances, cam_map, active_cams = tracker.get_instances(fg, session.cameras)

In [ ]:
# from scipy.optimize import linear_sum_assignment
importlib.reload(geometry)
importlib.reload(tracker)

# inst1 = cam_instances['back'][1]
# inst2 = cam_instances['midL'][1]
# cache=tracker.TrackerCache()
# # geometry.calc_epipolar_score(
# #     inst1, cam_map['back'],
# #     inst2, cam_map['midL'],
# #     cache
# # )

error = geometry.calc_reprojection_score(
    inst1, cam_map['back'],
    inst2, cam_map['midL'],
    cache
)
error

# test triangulation and reprojection
# gp = {'back': inst1, 'midL': inst2}
# X = geometry.triangulate_group(gp, cam_map, cache)
# reprojs = geometry.reproject_points(X, cache.getP(cam_map['back']))
# reproj_error = geometry.instance_pixel_distance(reprojs, inst1)
fg = session.frame_group(67)
cost = tracker.match_frame(fg, session, cache, num_animals=4)
cost
#

In [ ]:
import luc3d_tracker_helper as lt
import importlib; importlib.reload(lt)
# lt.triangulate_group(gp, cam_map, caches=lt.TrackerCaches())[0]
cam_instances, cam_map, active_cams = lt.collect_instances(fg, session.cameras)
# cost3 = lt.match_pairwise(cam_instances, cam_map, active_cams, num_animals=4, prev_assignments=None, caches=lt.TrackerCaches())
groups = lt.match_pairwise(fg, session, caches=lt.TrackerCaches(), num_animals=4)
groups

In [ ]:
cost == groups